- D-Fire RF-DETR-L | Kaggle T4 x2
  - Input: Kaggle Dataset dfire-relabeled, tạo từ D-Fire.zip
  - Accelerator: GPU T4 x2; Internet on; không dùng P100
  - Protocol: COCO pretrained, split cố định trong ZIP, seed 20260707, 640 px, 20 epoch
  - Global batch: 4 ảnh/T4 x 2 T4 x accumulation 4 = effective batch 32
  - Augmentation chung: horizontal flip 0.5; tắt augmentation khác
  - Resume: tự quét working + Kaggle Input; Input cần resume_protocol.json cùng run checkpoint
  - Output: /kaggle/working/runs/dfire_rfdetr_large


In [ ]:
!find /kaggle/input/datasets -maxdepth 5 -type d | head -80


In [ ]:
from pathlib import Path

DATASET_SLUG = 'dfire-relabeled'
SEED = 20260707
EPOCHS = 20
RESOLUTION = 640

INPUT_BASE = Path('/kaggle/input/datasets')
WORK_ROOT = Path('/kaggle/working')
RUN_NAME = 'dfire_rfdetr_large'
MICRO_BATCH = 4
GRAD_ACCUM = 4
RUN_DIR = WORK_ROOT / 'runs' / RUN_NAME


In [ ]:
import subprocess
import sys

subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rfdetr[train]==1.8.3'], check=True)

import importlib.metadata
import torch

devices = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
capabilities = [torch.cuda.get_device_capability(index) for index in range(torch.cuda.device_count())]
cuda_version = tuple(int(part) for part in torch.version.cuda.split('.')[:2])
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'rfdetr': importlib.metadata.version('rfdetr'), 'devices': devices, 'capabilities': capabilities})
assert len(devices) == 2 and all('T4' in name for name in devices), f'Cần Accelerator GPU T4 x2, hiện có {devices}'
assert all(value >= (7, 0) for value in capabilities), f'GPU không được torch {torch.__version__} hỗ trợ: {capabilities}'
assert cuda_version >= (12, 8), f'Cần Kaggle CUDA runtime >= 12.8, hiện có {torch.version.cuda}'


In [ ]:
from collections.abc import Mapping
import json, os, shutil, tempfile
PROTOCOL={'schema':1,'framework':'rfdetr-1.8.3','model':'RFDETRLarge','resolution':RESOLUTION,'epochs':EPOCHS,'seed':SEED,'micro_batch':MICRO_BATCH,'grad_accum':GRAD_ACCUM,'devices':2}; protocol=RUN_DIR/'resume_protocol.json'; existed=RUN_DIR.exists()
dataset_candidates=list(INPUT_BASE.glob(f'*/{DATASET_SLUG}/{DATASET_SLUG}/D-Fire'))
if len(dataset_candidates)!=1: raise FileNotFoundError(dataset_candidates)
data_root=dataset_candidates[0]
def valid(p,source):
 state=torch.load(p,map_location='cpu',weights_only=False)
 if not isinstance(state,Mapping) or not isinstance(state.get('epoch'),int) or not isinstance(state.get('global_step'),int): raise ValueError('epoch/global_step')
 if not isinstance(state.get('state_dict'),Mapping) or not state['state_dict'] or not state.get('optimizer_states') or not state.get('lr_schedulers') or not isinstance(state.get('loops',{}).get('fit_loop'),Mapping): raise ValueError('không full-state')
 q=protocol if source=='working' else next((x/'resume_protocol.json' for x in (p.parent,*p.parents) if (x/'resume_protocol.json').is_file()),None)
 if q is None or json.loads(q.read_text())!=PROTOCOL: raise ValueError('protocol')
 return {'path':p,'source':source,'epoch':state['epoch'],'step':state['global_step']}
def scan(paths,source):
 out=[]
 for p in paths:
  try: out.append(valid(p,source))
  except Exception as e: print('reject',source,p,e)
 return out
if existed and not protocol.is_file(): raise RuntimeError('working thiếu protocol')
w=scan(RUN_DIR.glob('checkpoint_*.ckpt'),'working')
if existed and not w: raise RuntimeError('working không checkpoint hợp lệ')
i=scan(INPUT_BASE.rglob('checkpoint_*.ckpt'),'input'); chosen=max(w+i,key=lambda x:(x['epoch'],x['step'],x['source']=='working'),default=None)
if chosen and chosen['source']=='input':
 d=RUN_DIR/'input_epoch_{:03d}_step_{:09d}.ckpt'.format(chosen['epoch'],chosen['step']); d.parent.mkdir(parents=True,exist_ok=True); t=d.with_suffix('.tmp'); shutil.copyfile(chosen['path'],t); os.replace(t,d); resume_path=d
else: resume_path=chosen['path'] if chosen else None
if not chosen and not existed: RUN_DIR.mkdir(parents=True,exist_ok=True); protocol.write_text(json.dumps(PROTOCOL,sort_keys=True))
run_complete=bool(chosen and chosen['epoch']>=EPOCHS-1); print({'resume_source':chosen['source'] if chosen else 'fresh','resume_path':str(resume_path) if resume_path else None,'complete':run_complete})


In [ ]:
from rfdetr import RFDETRLarge
if run_complete:
    print(f'Không train lại: checkpoint đã hoàn thành epoch {EPOCHS}.')
else:
    model = RFDETRLarge(resolution=RESOLUTION)
    model.train(dataset_dir=str(data_root), dataset_file='yolo', output_dir=str(RUN_DIR), epochs=EPOCHS, batch_size=MICRO_BATCH, grad_accum_steps=GRAD_ACCUM, accelerator='gpu', devices=2, strategy='ddp_notebook', amp_dtype='fp16', num_workers=2, checkpoint_interval=1, seed=SEED, early_stopping=False, tensorboard=False, multi_scale=False, aug_config={'HorizontalFlip': {'p': 0.5}}, warmup_epochs=0.0, lr_scheduler='cosine', lr_min_factor=0.01, resume=str(resume_path) if resume_path else None)


In [ ]:
checkpoints = sorted(RUN_DIR.glob('checkpoint_*.ckpt'))
weights = sorted(RUN_DIR.glob('*.pth'))
for path in checkpoints + weights:
    print(path, path.stat().st_size)
assert checkpoints
assert weights
